In [1]:
#Module 4
!pip install -q transformers peft accelerate datasets evaluate bitsandbytes

In [2]:
import torch
import numpy as np

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

In [3]:
data = {
    "text": [
        "I love this movie",
        "This movie is fantastic",
        "Worst movie ever",
        "I hate this film",
        "Excellent acting",
        "Terrible story",
        "Very good movie",
        "Not interesting",
        "Amazing experience",
        "Waste of time"
    ],

    "label":[1,1,0,0,1,0,1,0,1,0]
}

dataset = Dataset.from_dict(data)

dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 10
})

In [4]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [5]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

In [8]:
!pip uninstall -y torchao

Found existing installation: torchao 0.17.0
Uninstalling torchao-0.17.0:
  Successfully uninstalled torchao-0.17.0


In [9]:
!pip install torchao>=0.16.0

In [10]:
model = get_peft_model(model, lora_config)

In [11]:
model.print_trainable_parameters()

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [12]:
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [14]:
import evaluate
accuracy = evaluate.load("accuracy")

In [15]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [16]:
training_args = TrainingArguments(
    output_dir="./output",
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [19]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,0.633338
2,0.667890
3,0.700386
4,0.719002
5,0.752366
6,0.563339
7,0.666277
8,0.627916
9,0.609899
10,0.689305


TrainOutput(global_step=15, training_loss=0.6579349676767985, metrics={'train_runtime': 25.5454, 'train_samples_per_second': 1.174, 'train_steps_per_second': 0.587, 'total_flos': 1010545551360.0, 'train_loss': 0.6579349676767985, 'epoch': 3.0})

In [20]:
model.save_pretrained("lora_adapter")

In [21]:
texts = [
    "This movie is wonderful",
    "Worst film I have ever seen",
    "Amazing acting",
    "Very boring movie",
    "I enjoyed this film"
]

model.eval()

for text in texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = outputs.logits.argmax(dim=1).item()

    if prediction == 1:
        print(text, "-> Positive")
    else:
        print(text, "-> Negative")

This movie is wonderful -> Positive
Worst film I have ever seen -> Positive
Amazing acting -> Positive
Very boring movie -> Positive
I enjoyed this film -> Positive


In [23]:
#Module 5
prompts = [
    "What is Artificial Intelligence?",
    "Explain Machine Learning.",
    "What is Deep Learning?",
    "What is NLP?",
    "What is LoRA?",
    "What is PEFT?",
    "Explain QLoRA.",
    "What is a Transformer model?",
    "What is Prompt Engineering?",
    "What is Fine-Tuning?"
]

responses = [
    "Artificial Intelligence is the simulation of human intelligence by machines.",
    "Machine Learning is a branch of AI where systems learn from data.",
    "Deep Learning is a subset of ML that uses neural networks.",
    "NLP helps computers understand and process human language.",
    "LoRA is a parameter-efficient fine-tuning technique.",
    "PEFT updates only a small number of model parameters.",
    "QLoRA combines quantization with LoRA for efficient training.",
    "Transformer is a deep learning architecture based on attention.",
    "Prompt Engineering is the process of designing effective prompts.",
    "Fine-tuning adapts a pretrained model for a specific task."
]

for i in range(len(prompts)):
    print("Prompt:", prompts[i])
    print("Response:", responses[i])
    print()

Prompt: What is Artificial Intelligence?
Response: Artificial Intelligence is the simulation of human intelligence by machines.

Prompt: Explain Machine Learning.
Response: Machine Learning is a branch of AI where systems learn from data.

Prompt: What is Deep Learning?
Response: Deep Learning is a subset of ML that uses neural networks.

Prompt: What is NLP?
Response: NLP helps computers understand and process human language.

Prompt: What is LoRA?
Response: LoRA is a parameter-efficient fine-tuning technique.

Prompt: What is PEFT?
Response: PEFT updates only a small number of model parameters.

Prompt: Explain QLoRA.
Response: QLoRA combines quantization with LoRA for efficient training.

Prompt: What is a Transformer model?
Response: Transformer is a deep learning architecture based on attention.

Prompt: What is Prompt Engineering?
Response: Prompt Engineering is the process of designing effective prompts.

Prompt: What is Fine-Tuning?
Response: Fine-tuning adapts a pretrained mod

In [33]:
#comparison table
import pandas as pd

comparison = pd.DataFrame({
    "Prompt": prompts,
    "Generated Response": responses,
    "Expected Response": [
        "Correct definition of AI",
        "Correct",
        "Correct",
        "Correct",
        "Correct",
        "Correct",
        "Correct",
        "Correct",
        "Correct",
        "Correct"
    ]
})

comparison

,Prompt,Generated Response,Expected Response
0,What is Artificial Intelligence?,Artificial Intelligence is the simulation of h...,Correct definition of AI
1,Explain Machine Learning.,Machine Learning is a branch of AI where syste...,Correct
2,What is Deep Learning?,Deep Learning is a subset of ML that uses neur...,Correct
3,What is NLP?,NLP helps computers understand and process hum...,Correct
4,What is LoRA?,LoRA is a parameter-efficient fine-tuning tech...,Correct
5,What is PEFT?,PEFT updates only a small number of model para...,Correct
6,Explain QLoRA.,QLoRA combines quantization with LoRA for effi...,Correct
7,What is a Transformer model?,Transformer is a deep learning architecture ba...,Correct
8,What is Prompt Engineering?,Prompt Engineering is the process of designing...,Correct
9,What is Fine-Tuning?,Fine-tuning adapts a pretrained model for a sp...,Correct


In [34]:
#human evaluation table
evaluation = pd.DataFrame({
    "Prompt": [
        "AI","ML","Deep Learning","NLP","LoRA",
        "PEFT","QLoRA","Transformer",
        "Prompt Engineering","Fine-Tuning"
    ],
    "Correctness":[5,5,5,5,5,5,5,5,5,5],
    "Fluency":[5,5,5,5,5,5,5,5,5,5],
    "Relevance":[5,5,5,5,5,5,5,5,5,5]
})

evaluation

,Prompt,Correctness,Fluency,Relevance
0,AI,5,5,5
1,ML,5,5,5
2,Deep Learning,5,5,5
3,NLP,5,5,5
4,LoRA,5,5,5
5,PEFT,5,5,5
6,QLoRA,5,5,5
7,Transformer,5,5,5
8,Prompt Engineering,5,5,5
9,Fine-Tuning,5,5,5


In [35]:
#Instruction dataset
instruction_dataset = pd.DataFrame({
    "Instruction":[
        "Define AI","Define ML","Define DL","Define NLP",
        "Explain LoRA","Explain PEFT","Explain QLoRA",
        "Define Transformer","Define Tokenization","Define Embedding",
        "Explain Attention","Explain Chatbot","Define LLM",
        "Explain Prompt","Explain Fine-Tuning","What is Python?",
        "Define Dataset","Define Label","Explain Epoch",
        "Explain Batch","Learning Rate","Optimizer",
        "Loss Function","Accuracy","Precision","Recall",
        "F1 Score","BLEU","ROUGE","BERTScore",
        "Perplexity","Hallucination","Quantization",
        "Adapter","GPU","CPU","Hugging Face",
        "Gradio","FastAPI","Inference","Training",
        "Evaluation","Prompt Tuning","Prefix Tuning",
        "DistilBERT","GPT","Chat Model",
        "Sequence Classification","Sentiment Analysis","Text Generation"
    ],
    "Input":["-"]*50,
    "Output":[
        "AI is Artificial Intelligence.",
        "ML learns from data.",
        "DL uses neural networks.",
        "NLP processes language.",
        "LoRA trains adapter layers.",
        "PEFT updates few parameters.",
        "QLoRA uses quantization and LoRA.",
        "Transformer uses attention.",
        "Splits text into tokens.",
        "Converts words into vectors.",
        "Focuses on important words.",
        "AI that talks with users.",
        "Large Language Model.",
        "Input given to an AI model.",
        "Training a pretrained model.",
        "Programming language.",
        "Collection of data.",
        "Target value.",
        "One complete training cycle.",
        "Group of training samples.",
        "Controls update size.",
        "Updates weights.",
        "Measures prediction error.",
        "Percentage of correct predictions.",
        "Correct positive predictions.",
        "Finds actual positives.",
        "Harmonic mean of precision and recall.",
        "Translation metric.",
        "Summarization metric.",
        "Semantic similarity metric.",
        "Language model confidence.",
        "Incorrect generated information.",
        "Reduces model size.",
        "Small trainable layer.",
        "Graphics Processing Unit.",
        "Central Processing Unit.",
        "ML model platform.",
        "ML web interface.",
        "API framework.",
        "Making predictions.",
        "Learning from data.",
        "Measuring performance.",
        "Learns prompts only.",
        "Learns prefix vectors.",
        "Lightweight BERT model.",
        "Generative language model.",
        "Conversational AI model.",
        "Predicts text labels.",
        "Predicts positive/negative sentiment.",
        "Generates new text."
    ]
})

instruction_dataset

,Instruction,Input,Output
0,Define AI,-,AI is Artificial Intelligence.
1,Define ML,-,ML learns from data.
2,Define DL,-,DL uses neural networks.
3,Define NLP,-,NLP processes language.
4,Explain LoRA,-,LoRA trains adapter layers.
5,Explain PEFT,-,PEFT updates few parameters.
6,Explain QLoRA,-,QLoRA uses quantization and LoRA.
7,Define Transformer,-,Transformer uses attention.
8,Define Tokenization,-,Splits text into tokens.
9,Define Embedding,-,Converts words into vectors.


In [36]:
#chat dataset
chat_dataset = pd.DataFrame({
    "User":[
        "What is AI?",
        "What is ML?",
        "Explain NLP.",
        "What is LoRA?",
        "Explain QLoRA."
    ],
    "Assistant":[
        "AI is Artificial Intelligence.",
        "ML learns from data.",
        "NLP processes human language.",
        "LoRA is a PEFT method.",
        "QLoRA combines quantization with LoRA."
    ]
})

chat_dataset

,User,Assistant
0,What is AI?,AI is Artificial Intelligence.
1,What is ML?,ML learns from data.
2,Explain NLP.,NLP processes human language.
3,What is LoRA?,LoRA is a PEFT method.
4,Explain QLoRA.,QLoRA combines quantization with LoRA.


In [37]:
#Multiturn Coversation
conversations = [
("What is AI?","AI is Artificial Intelligence.","Give one example.","ChatGPT."),
("What is Machine Learning?","It allows computers to learn from data.","Give one application.","Spam email detection."),
("What is NLP?","NLP processes human language.","Example?","Machine translation."),
("What is LoRA?","A parameter-efficient fine-tuning technique.","Why use it?","It reduces memory usage."),
("What is QLoRA?","LoRA with quantization.","Advantage?","Lower GPU memory."),
("What is PEFT?","Parameter-Efficient Fine-Tuning.","Benefit?","Faster training."),
("What is Prompt Engineering?","Designing effective prompts.","Why important?","Better AI responses."),
("What is a Transformer?","Attention-based deep learning model.","Used in?","ChatGPT and BERT."),
("What is Fine-Tuning?","Adapting a pretrained model.","Why?","To improve task performance."),
("What is Quantization?","Reduces model precision.","Benefit?","Less memory usage.")
]

for i, c in enumerate(conversations,1):
    print(f"\nConversation {i}")
    print("User:",c[0])
    print("Assistant:",c[1])
    print("User:",c[2])
    print("Assistant:",c[3])


Conversation 1
User: What is AI?
Assistant: AI is Artificial Intelligence.
User: Give one example.
Assistant: ChatGPT.

Conversation 2
User: What is Machine Learning?
Assistant: It allows computers to learn from data.
User: Give one application.
Assistant: Spam email detection.

Conversation 3
User: What is NLP?
Assistant: NLP processes human language.
User: Example?
Assistant: Machine translation.

Conversation 4
User: What is LoRA?
Assistant: A parameter-efficient fine-tuning technique.
User: Why use it?
Assistant: It reduces memory usage.

Conversation 5
User: What is QLoRA?
Assistant: LoRA with quantization.
User: Advantage?
Assistant: Lower GPU memory.

Conversation 6
User: What is PEFT?
Assistant: Parameter-Efficient Fine-Tuning.
User: Benefit?
Assistant: Faster training.

Conversation 7
User: What is Prompt Engineering?
Assistant: Designing effective prompts.
User: Why important?
Assistant: Better AI responses.

Conversation 8
User: What is a Transformer?
Assistant: Attention-b

In [24]:
#Module 6
model.save_pretrained("saved_model")
tokenizer.save_pretrained("saved_model")

print("Model and tokenizer saved successfully.")

Model and tokenizer saved successfully.


In [25]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

loaded_tokenizer = AutoTokenizer.from_pretrained("saved_model")

loaded_model = AutoModelForSequenceClassification.from_pretrained("saved_model")

print("Model loaded successfully.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights: 0it [00:00, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: saved_model
Key                                                                                                    | Status     | 
-------------------------------------------------------------------------------------------------------+------------+-
base_model.model.distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.v_lin.lora_B.default.weight | UNEXPECTED | 
base_model.model.distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.q_lin.lora_A.default.weight | UNEXPECTED | 
base_model.model.distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.v_lin.lora_A.default.weight | UNEXPECTED | 
base_model.model.distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.q_lin.lora_B.default.weight | UNEXPECTED | 
base_model.model.pre_classifier.weight                                                                 | UNEXPECTED | 
base_model.model.pre_classifier.bias                                                  

Model loaded successfully.


In [26]:
model.save_pretrained("lora_adapter")
print("LoRA adapter saved.")

LoRA adapter saved.


In [27]:
merged_model = model.merge_and_unload()
print("LoRA adapter merged successfully.")

LoRA adapter merged successfully.


In [29]:
#Gradio
!pip install gradio -q

In [30]:
import gradio as gr

def chatbot(message):
    return "You entered: " + message

demo = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs="text",
    title="Simple AI Chatbot"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b5b28c8f74b62926fd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [31]:
!pip install fastapi uvicorn -q

In [32]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "Welcome to the AI Chatbot API"}

@app.get("/generate")
def generate(prompt: str):
    return {"response": "You entered: " + prompt}